<a href="https://colab.research.google.com/github/sr606/Python-Practice/blob/main/Mermaid10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
✅ Column-level aware

✅ Join column mapping visible

✅ Smart section skipping

✅ No artificial truncation

✅ Clean rendering

✅ With full DOT printed to console

✅ Chunk-safe

Below is your complete upgraded architecture + updated scripts.

🏗 UPDATED ARCHITECTURE
agent.py
│
├── parsing/
│     ├── parser.py
│     ├── extractor.py
│     └── lineage_order.py
│
├── llm/
│     ├── llm_normalizer.py   ← upgraded
│     └── llm_dot_compiler.py ← upgraded
│
└── rendering/
      └── dot_sanitizer.py
🔥 1️⃣ UPDATE: llm_normalizer.py

We will:

Extract join column pairs

Detect null handling

Detect CASE derivations

Preserve full logic

Deduplicate cleanly

✅ REPLACE extract_joins() WITH THIS
def extract_join_columns(condition):
    """
    Extract column pairs from join condition
    Example:
    vor.registration_number = v.registration_no
    """
    pairs = re.findall(
        r"([a-zA-Z0-9_\.]+)\s*=\s*([a-zA-Z0-9_\.]+)",
        condition
    )
    return pairs


def extract_joins(sql_text):
    if not sql_text:
        return []

    join_pattern = re.findall(
        r"""
        \b
        (LEFT|RIGHT|INNER|FULL)?      # Join type
        \s*
        (OUTER\s+)?
        JOIN
        \s+
        (
            \([^\)]*?\)
            |
            [a-zA-Z0-9_\.]+
        )
        \s+
        .*?
        \bON\b
        \s+
        (.*?)
        (?=
            \bLEFT\b|
            \bRIGHT\b|
            \bINNER\b|
            \bFULL\b|
            \bWHERE\b|
            \bGROUP\b|
            \bORDER\b|
            $
        )
        """,
        sql_text,
        re.IGNORECASE | re.DOTALL | re.VERBOSE
    )

    joins = []

    for jt, outer, table, condition in join_pattern:
        join_type = (jt.upper() if jt else "JOIN") + " JOIN"

        joins.append({
            "join_type": join_type,
            "table": table.strip(),
            "condition": condition.strip(),
            "columns": extract_join_columns(condition)
        })

    return joins
✅ UPDATE extract_transformations()

Replace with:

def extract_transformations(transform_text):
    results = []

    for line in transform_text.split("\n"):
        line = line.strip()

        if "=" in line and not line.startswith("--"):
            parts = line.split("=", 1)

            target = parts[0].strip()
            logic = parts[1].strip()

            t_type = "DIRECT"

            if "NVL(" in logic.upper():
                t_type = "NULL_HANDLING"
            elif "CASE" in logic.upper():
                t_type = "DERIVATION"

            results.append({
                "target": target,
                "logic": logic,
                "type": t_type
            })

    return results
🔥 2️⃣ UPDATE: llm_dot_compiler.py

We now:

Remove truncation

Dynamically skip empty sections

Show join column mappings

Show transformation source → target

Print full DOT

✅ REPLACE generate_dot() WITH THIS
def generate_dot(workflow_name, stages):

    # DO NOT TRUNCATE ANYTHING
    minimal_stages = []

    for s in stages:
        minimal_stages.append({
            "stage_id": s["stage_id"],
            "inputs": s.get("inputs", []),
            "outputs": s.get("outputs", []),
            "primary_tables": s.get("primary_tables", []),
            "joins": s.get("joins", []),
            "filters": s.get("filters", []),
            "transformations": s.get("transformations", [])
        })

    payload = json.dumps({
        "workflow_name": workflow_name,
        "stages": minimal_stages
    }, ensure_ascii=False)

    enhanced_prompt = PROMPT + """

SMART RULES:

- If primary_tables empty → DO NOT render section
- If joins empty → DO NOT render section
- If transformations empty → DO NOT render section

JOINS SECTION:
For each join:
• Show join type
• Show table name
• Show column pairs like:
    columnA = columnB

TRANSFORMATIONS SECTION:
For each transformation:
• Show: target ← logic
• If type = NULL_HANDLING, prefix with (Null Handling)
• If type = DERIVATION, prefix with (Derived)

Do NOT truncate.
Do NOT hallucinate.
Do NOT add markdown.
"""

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0,
        messages=[
            {"role": "system", "content": "You generate strictly valid Graphviz DOT."},
            {"role": "user", "content": enhanced_prompt + "\n\nWorkflow JSON:\n" + payload}
        ]
    )

    dot_output = response.choices[0].message.content.strip()

    print("\n================ FULL DOT OUTPUT ================")
    print(dot_output)
    print("=================================================\n")

    return dot_output
🔥 3️⃣ UPDATE agent.py (Print DOT Before Render)

Inside process_file() change:

dot_code = generate_dot(workflow_name, structured_stages)

dot_code = clean_and_validate_dot(dot_code)

To:

dot_code = generate_dot(workflow_name, structured_stages)

print("\n=========== CLEANED DOT ===========")
print(dot_code)
print("===================================\n")

dot_code = clean_and_validate_dot(dot_code)
🔥 4️⃣ OPTIONAL: Improve DOT PROMPT Header

Replace your existing PROMPT with this improved version:

PROMPT = """
You are a professional ETL lineage Graphviz compiler.

Generate COMPLETE valid DOT code.

STRICT RULES:

- rankdir=LR
- bgcolor="white"
- node [shape=plain, fontname="Helvetica"]
- Use subgraph cluster_workflow
- Do NOT use markdown
- Do NOT use ```
- Do NOT rename stage_id
- Do NOT remove any joins or transformations
- Skip empty sections automatically

Node format must use table HTML style.

Output must start with:

digraph WORKFLOW_NAME {
"""
🎯 WHAT YOU WILL NOW SEE

Each stage will look like:

ORA_Ext_Vehicle_Off_Road_Data

Primary Tables
• supplierdata_ext.vor_vehicle_off_road

Joins
• LEFT JOIN vehicle_dim
  registration_number = registration_no

Transformations
• (Null Handling) ORIGINAL_WORKSHEET_NUMBER ← NVL(...)
• (Derived) SUPP_SK ← CASE WHEN ...

If no joins → joins section omitted
If no transformations → transformations section omitted

Clean. Intelligent. Professional.

🚀 WHAT THIS GIVES YOU

✔ Full SQL preserved
✔ Join column mapping
✔ Null handling labeled
✔ Derived columns labeled
✔ No truncation
✔ Dynamic clean layout
✔ Full DOT printed for debugging
✔ Ready for enterprise lineage

🧠 Next Step

Run this on your heavy ORA SQL stage.

If DOT is too large, next enhancement:

Auto-chunk stage-level rendering

Or collapse repeated joins